# Source Code:
*Note: Since mpi4py po yung gamit, kailangan po ng terminal command so yung output po for this is nasa second cell*

In [ ]:
# mpiexec -n 5 .venv\Scripts\python.exe PDC-Fin-Project-TeamBrackie.py 2>$null

import pandas as pd
import time
from mpi4py import MPI

# ── MPI Setup ─────────────────────────────────────────────────────────────────
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

CSV_PATH = r'C:\Users\Josh\Downloads\boc_lite_2019.csv'
NROWS    = 100_000

start_time = time.time()

# Timing lists
s_timepertask = []
p_timepertask = []
d_timepertask = []


# ──────────────────────────────────────────────────────────────────────────────
# SEQUENTIAL
# ──────────────────────────────────────────────────────────────────────────────
def sequential(df):
    print('=== Sequential Processing ===\n')

    # Task 1 – Filter
    print('> Task 1: Filter Data\n')
    t0 = time.time()
    s_filtered = df[df['countryorigin_iso3'] == 'CHN']
    elapsed = time.time() - t0
    s_timepertask.append(elapsed)
    print(s_filtered.head(5))
    print(f'\n> Task done in {elapsed:.4f} seconds.\n')
    print('==============')

    # Task 2 – Aggregate
    print('> Task 2: Aggregate Data\n')
    t0 = time.time()
    s_agg = df.groupby('countryorigin_iso3').agg({'dutiablevaluephp': 'mean'}).reset_index()
    elapsed = time.time() - t0
    s_timepertask.append(elapsed)
    print(s_agg)
    print(f'\n> Task done in {elapsed:.4f} seconds.\n')
    print('==============')

    # Task 3 – Sort
    print('> Task 3: Sort Data\n')
    t0 = time.time()
    s_sorted = df.sort_values(by='dutiablevaluephp', ascending=False)
    elapsed = time.time() - t0
    s_timepertask.append(elapsed)
    print(s_sorted.head(5))
    print(f'\n> Task done in {elapsed:.4f} seconds.\n')
    print('==============\n')

    print(f'Sequential Processing Time Total: {sum(s_timepertask):.4f} seconds.')


# ──────────────────────────────────────────────────────────────────────────────
# PARALLEL  (MPI)
# ──────────────────────────────────────────────────────────────────────────────
def parallel(df=None):
    if rank == 0:
        print('\n=== Parallel Processing ===\n')

        workers = size - 1
        total_rows = len(df)
        rows_per_worker = total_rows // workers

        # Distribute chunks
        for i in range(1, size):
            s = (i - 1) * rows_per_worker
            e = s + rows_per_worker if i < size - 1 else total_rows
            comm.send(df.iloc[s:e].copy(), dest=i, tag=0)

        # Task 1 – Filter
        print('> Task 1: Filter Data\n')
        t0 = time.time()
        parts = [comm.recv(source=i, tag=1) for i in range(1, size)]
        p_filtered = pd.concat(parts, ignore_index=True)
        elapsed = time.time() - t0
        p_timepertask.append(elapsed)
        print(p_filtered.head(5))
        print(f'\n> Task done in {elapsed:.4f} seconds.\n')
        print('==============')

        # Task 2 – Aggregate
        print('> Task 2: Aggregate Data\n')
        t0 = time.time()
        parts = [comm.recv(source=i, tag=2) for i in range(1, size)]
        p_agg = pd.concat(parts, ignore_index=True) \
                  .groupby('countryorigin_iso3').agg({'dutiablevaluephp': 'mean'}).reset_index()
        elapsed = time.time() - t0
        p_timepertask.append(elapsed)
        print(p_agg)
        print(f'\n> Task done in {elapsed:.4f} seconds.\n')
        print('==============')

        # Task 3 – Sort
        print('> Task 3: Sort Data\n')
        t0 = time.time()
        parts = [comm.recv(source=i, tag=3) for i in range(1, size)]
        p_sorted = pd.concat(parts, ignore_index=True) \
                     .sort_values(by='dutiablevaluephp', ascending=False)
        elapsed = time.time() - t0
        p_timepertask.append(elapsed)
        print(p_sorted.head(5))
        print(f'\n> Task done in {elapsed:.4f} seconds.\n')
        print('==============\n')

        print(f'Parallel Processing Time Total: {sum(p_timepertask):.4f} seconds.')

    else:
        local_df = comm.recv(source=0, tag=0)

        local_filtered = local_df[local_df['countryorigin_iso3'] == 'CHN']
        comm.send(local_filtered, dest=0, tag=1)
        print(f'Processor {rank} has done Task 1')

        local_agg = local_df.groupby('countryorigin_iso3') \
                             .agg({'dutiablevaluephp': 'mean'}).reset_index()
        comm.send(local_agg, dest=0, tag=2)
        print(f'Processor {rank} has done Task 2')

        local_sorted = local_df.sort_values(by='dutiablevaluephp', ascending=False)
        comm.send(local_sorted, dest=0, tag=3)
        print(f'Processor {rank} has done Task 3')


# ──────────────────────────────────────────────────────────────────────────────
# DISTRIBUTED  (PySpark)
# ──────────────────────────────────────────────────────────────────────────────
def distributed(df):
    print('\n=== Distributed Processing (PySpark) ===\n')

    try:
        from pyspark.sql import SparkSession
        from pyspark.sql import functions as F
    except ImportError:
        print('[SKIP] PySpark not installed.  Run: pip install pyspark\n')
        return

    import os, sys, warnings
    warnings.filterwarnings('ignore')

    venv_python = os.path.abspath(sys.executable)
    os.environ.setdefault('HADOOP_HOME', r'C:\hadoop')
    os.environ['PYSPARK_PYTHON'] = venv_python
    os.environ['PYSPARK_DRIVER_PYTHON'] = venv_python

    spark = (
        SparkSession.builder
        .appName('PDC-Final-TeamBrackie')
        .master('local[*]')
        .config('spark.driver.memory', '2g')
        .config('spark.sql.shuffle.partitions', str((size - 1) * 2))
        .config('spark.log.structuredLogging.enabled', 'false')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel('ERROR')
    log4j = spark.sparkContext._jvm.org.apache.log4j
    log4j.Logger.getLogger('org.apache.spark.util.ShutdownHookManager').setLevel(log4j.Level.OFF)
    log4j.Logger.getLogger('org.apache.spark.SparkContext').setLevel(log4j.Level.OFF)

    print(f'Spark {spark.version}  |  Master: {spark.sparkContext.master}'
          f'  |  Cores: {spark.sparkContext.defaultParallelism}\n')

    t_load = time.time()
    sdf = spark.createDataFrame(df).cache()
    sdf.count()
    print(f'DataFrame created in {time.time() - t_load:.4f} seconds.\n')

    # Task 1 – Filter
    print('> Task 1: Filter Data\n')
    t0 = time.time()
    d_filtered = sdf.filter(F.col('countryorigin_iso3') == 'CHN')
    d_filtered.count()
    elapsed = time.time() - t0
    d_timepertask.append(elapsed)
    print(d_filtered.limit(5).toPandas().head(5))
    print(f'\n> Task done in {elapsed:.4f} seconds.\n')
    print('==============')

    # Task 2 – Aggregate
    print('> Task 2: Aggregate Data\n')
    t0 = time.time()
    d_agg = sdf.groupBy('countryorigin_iso3') \
               .agg(F.mean('dutiablevaluephp').alias('dutiablevaluephp'))
    d_agg.count()
    elapsed = time.time() - t0
    d_timepertask.append(elapsed)
    print(d_agg.orderBy('countryorigin_iso3').toPandas())
    print(f'\n> Task done in {elapsed:.4f} seconds.\n')
    print('==============')

    # Task 3 – Sort
    print('> Task 3: Sort Data\n')
    t0 = time.time()
    d_sorted = sdf.orderBy(F.col('dutiablevaluephp').desc())
    d_sorted.count()
    elapsed = time.time() - t0
    d_timepertask.append(elapsed)
    print(d_sorted.limit(5).toPandas().head(5))
    print(f'\n> Task done in {elapsed:.4f} seconds.\n')
    print('==============\n')

    print(f'Distributed Processing Time Total: {sum(d_timepertask):.4f} seconds.')
    spark.stop()


# ──────────────────────────────────────────────────────────────────────────────
# COMPARISON + DISCUSSION
# ──────────────────────────────────────────────────────────────────────────────
def print_comparison():
    labels = ['Filter', 'Aggregate', 'Sort']
    print('\n=== Performance Comparison ===\n')
    print(f"{'Task':<12} {'Sequential':>14} {'Parallel(MPI)':>15} {'Distributed(Spark)':>20}")
    print('-' * 62)
    for i, label in enumerate(labels):
        s = f"{s_timepertask[i]:.4f}s" if i < len(s_timepertask) else 'N/A'
        p = f"{p_timepertask[i]:.4f}s" if i < len(p_timepertask) else 'N/A'
        d = f"{d_timepertask[i]:.4f}s" if i < len(d_timepertask) else 'N/A'
        print(f'{label:<12} {s:>14} {p:>15} {d:>20}')
    print('-' * 62)
    s_t = sum(s_timepertask)
    p_t = sum(p_timepertask)
    d_t = sum(d_timepertask)
    print(f"{'TOTAL':<12} {s_t:>13.4f}s {p_t:>14.4f}s {d_t:>19.4f}s")
    if s_t and p_t:
        print(f'\nMPI Speedup   vs Sequential : {s_t / p_t:.2f}x')
    if s_t and d_t:
        print(f'Spark Speedup vs Sequential : {s_t / d_t:.2f}x')
    if p_t and d_t:
        print(f'Spark Speedup vs MPI        : {p_t / d_t:.2f}x')


def print_discussion():
    print("""
=== Fault Tolerance & Scalability Discussion ===

Sequential
  - No fault tolerance. A crash loses all progress.
  - Bounded by a single CPU core; does not scale.

Parallel - MPI (mpi4py)
  - No built-in recovery. If one worker crashes, the whole job fails.
  - Mitigation: checkpoint intermediate results to disk and restart.
  - Scales well on multi-core machines and HPC clusters.
  - Communication overhead grows with process count (Amdahl's Law).

Distributed - PySpark
  - Built-in fault tolerance via RDD lineage (DAG).
    If a partition is lost, Spark recomputes only that partition
    from its parent -- no full restart needed.
  - Checkpointing (rdd.checkpoint() or writing to Parquet/HDFS)
    cuts recomputation cost for long pipelines.
  - Designed for horizontal scaling: add worker nodes and Spark
    redistributes partitions automatically.
  - Best for large-scale ETL and cloud pipelines (EMR, Databricks).
  - Note: JVM startup overhead makes it slower than MPI on small
    datasets. Advantage shows at millions of rows on a real cluster.
""")


# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────
def main():
    if rank == 0:
        df = pd.read_csv(CSV_PATH, encoding='latin-1', nrows=NROWS)
        read_time = time.time() - start_time
        print(f'DataFrame created in {read_time:.4f} seconds.\n\n')
        sequential(df)

    comm.Barrier()

    if rank == 0:
        df = pd.read_csv(CSV_PATH, encoding='latin-1', nrows=NROWS)
        parallel(df)
    else:
        parallel()

    comm.Barrier()

    if rank == 0:
        df = pd.read_csv(CSV_PATH, encoding='latin-1', nrows=NROWS)
        distributed(df)
        print_comparison()
        print_discussion()
        print(f'{time.time() - start_time:.4f} seconds in total.')

main()

## **Output:**

In [2]:
import subprocess
result = subprocess.run(
    ["mpiexec", "-n", "5",
     r"C:\Users\Josh\Desktop\Josh Folder\School\Perpetual\3rd Year\Parallel and Distributed Computing\.venv\Scripts\python.exe",
     r"C:\Users\Josh\Desktop\Josh Folder\School\Perpetual\3rd Year\Parallel and Distributed Computing\PDC-Fin-Project-TeamBrackie.py"],
    capture_output=True, text=True
)
print(result.stdout)


Processor 4 has done Task 1
Processor 4 has done Task 2
Processor 4 has done Task 3
Processor 1 has done Task 1
Processor 1 has done Task 2
Processor 1 has done Task 3
Processor 3 has done Task 1
Processor 3 has done Task 2
Processor 3 has done Task 3
Processor 2 has done Task 1
Processor 2 has done Task 2
Processor 2 has done Task 3
Missing Python executable 'C:\Users\Josh\Desktop\Josh Folder\School\Perpetual\3rd Year\Parallel and Distributed Computing\.venv\Scripts\python.exe', defaulting to 'C:\Users\Josh\Desktop\Josh Folder\School\Perpetual\3rd Year\Parallel and Distributed Computing\.venv\Lib\site-packages\pyspark\bin\..' for SPARK_HOME environment variable. Please install Python or specify the correct Python executable in PYSPARK_DRIVER_PYTHON or PYSPARK_PYTHON environment variable to detect SPARK_HOME safely.
DataFrame created in 0.3588 seconds.


=== Sequential Processing ===

> Task 1: Filter Data

               uid    ty      tq  ... countryexport_iso3  subport  port
0  2019

## **Table:**

In [3]:
import re
import pandas as pd

seq_times  = re.findall(r'Task done in ([\d.]+) seconds', result.stdout[:result.stdout.find('=== Parallel')])
par_times  = re.findall(r'Task done in ([\d.]+) seconds', result.stdout[result.stdout.find('=== Parallel'):result.stdout.find('=== Distributed')])
dist_times = re.findall(r'Task done in ([\d.]+) seconds', result.stdout[result.stdout.find('=== Distributed'):])

tasks = ['Filter Data', 'Aggregate Data', 'Sort Data']

summary_df = pd.DataFrame({
    'Task': tasks,
    'Sequential (seconds)':   [float(t) for t in seq_times],
    'Parallel (seconds)':     [float(t) for t in par_times],
    'Distributed (seconds)':  [float(t) for t in dist_times],
})

totals = pd.DataFrame([{
    'Task': 'TOTAL',
    'Sequential (seconds)':  summary_df['Sequential (seconds)'].sum(),
    'Parallel (seconds)':    summary_df['Parallel (seconds)'].sum(),
    'Distributed (seconds)': summary_df['Distributed (seconds)'].sum(),
}])

summary_df = pd.concat([summary_df, totals], ignore_index=True)
summary_df


,Task,Sequential (seconds),Parallel (seconds),Distributed (seconds)
0,Filter Data,0.020,0.0589,0.6868
1,Aggregate Data,0.011,0.0100,1.0329
2,Sort Data,0.040,0.2100,0.2205
3,TOTAL,0.071,0.2789,1.9402


### **Technical Report**

1. **Problem Description**
- The primary objective of the program was to optimize the processing of a dataset (Bureau of Customs 2019 lite data) using distributed computing and measure whether parallelized and distributed approaches could significantly reduce execution time compared to a standard sequential Python script. The activity focused on three specific data operations:
  - **Filtering:** Extracting records where the country of origin is "CHN".
  - **Aggregation:** Calculating the mean `dutiablevaluephp` grouped by country.
  - **Sorting:** Organizing the dataset by dutiable value in descending order.

2. **Parallelization & Distribution Approach**
- **Parallel (MPI):** Utilizes the Master-Worker architecture using `mpi4py` with 5 processes. The master loads the DataFrame, partitions the data into chunks, distributes tasks to workers, and gathers the results. The four workers receive data chunks, perform local filtering, aggregation, and sorting, then send results back to the master.
- **Distributed (PySpark):** Uses Apache Spark in `local[*]` mode, which distributes work across all available CPU cores using Spark's internal DAG scheduler. The pandas DataFrame is converted to a Spark DataFrame, and all three tasks are executed as lazy transformations triggered by actions (`.count()`, `.toPandas()`). Spark handles partitioning and parallel execution automatically.

3. **Performance Analysis**

| # | Task | Sequential (seconds) | Parallel (seconds) | Distributed (seconds) |
| --- | ---- | -------------------- | ------------------ | --------------------- |
| 0 | Filter Data | 0.020 | 0.0589 | 0.6868 |
| 1 | Aggregate Data | 0.011 | 0.0100 | 1.0329 |
| 2 | Sort Data | 0.0400 | 0.2100 | 0.2205 |
| 3 | TOTAL | 0.071 | 0.2789 | 1.9402 |

- Based on the execution logs, the MPI parallel implementation showed performance gains on Filter and Aggregate tasks due to data being split across 4 workers processing simultaneously. Sort was slower in parallel due to MPI communication overhead. The master must serialize, send, receive, and re-merge chunks via `pd.concat` before doing a final `sort_values`, which costs more than a single sequential sort.
- PySpark showed higher per-task times on this 100,000-row dataset due to JVM startup cost and the overhead of converting a pandas DataFrame to a Spark DataFrame. On a real cluster with millions of rows, Spark's distributed execution and built-in fault tolerance would provide significant advantages over both sequential and MPI approaches.

4. **Fault Tolerance & Scalability**
- **Sequential:** No fault tolerance. A crash loses all progress. Bounded by a single CPU core.
- **MPI:** No built-in recovery. If one worker crashes, the entire process fails. Mitigation requires manual checkpointing to disk. Scales well on clusters of computer networks, but communication overhead grows with process count (Amdahl's Law).
- **PySpark:** Built-in fault tolerance via RDD lineage (DAG). If a partition is lost due to an executor crash, Spark automatically recomputes only that partition from its parent RDD, which means no full restart is needed. Designed for horizontal scaling: adding worker nodes causes Spark to redistribute partitions automatically. Best suited for large-scale ETL and cloud-native pipelines (AWS EMR, Databricks, GCP Dataproc).

5. **Challenges Encountered**
- Running `mpi4py` requires the `mpiexec` terminal command, which cannot be executed directly within standard Jupyter cells. `subprocess` was used to capture the output.
- PySpark on Windows requires `winutils.exe` and `HADOOP_HOME` to be set, as Spark uses Hadoop's file system utilities even in local mode. Additionally, PySpark 3.5.x is incompatible with Python 3.13 due to socket API changes, requiring a Python 3.11 virtual environment.
- The `comm.Barrier()` call was necessary to ensure the sequential baseline completed before parallel processes began, and again before the Spark session started.